In [1]:

from files.functions import *
import numpy as np
import pandas as pd
import warnings
from files.xgboostImplementation import XGBoost
import pickle
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
warnings.filterwarnings('ignore')

In [2]:
data = pd.read_csv(fullDataPath(COIN))
data = data.loc[:, ~data.columns.str.contains('^Unnamed:')]
merged = dataSetup(data)
merged

,BB_Lower,BB_Upper,value,avg_sentiment,volume,OBV,SMA_7,Volume_MA_7,low,high,open,close,gradient
time,,,,,,,,,,,,,
2015-07-20,268.225208,299.188792,58.0,0.000000,782.883420,-3.801571e+06,283.615714,4417.348637,277.37,280.00,277.98,280.00,0.00
2015-07-21,266.070954,300.106046,58.0,0.000000,4943.559434,-3.802354e+06,285.645714,5228.512882,276.85,281.27,279.96,277.32,-2.68
2015-07-22,263.804410,301.155590,58.0,0.000000,4687.909383,-3.797411e+06,288.280000,5345.414337,275.01,278.54,277.33,277.89,0.57
2015-07-23,263.024045,301.345955,58.0,0.000000,5306.919575,-3.802099e+06,290.037143,5421.928306,276.28,279.75,277.96,277.39,-0.50
2015-07-24,261.412051,301.912949,58.0,0.000000,7362.469083,-3.796792e+06,291.622857,5397.937160,276.43,291.52,277.23,289.12,11.73
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-07,101285.329040,115642.848960,52.0,-0.000980,4455.083530,-2.884451e+04,111430.271429,0.000000,107507.00,109741.64,109217.98,108269.84,-948.14
2025-07-08,101285.329040,115642.848960,50.0,0.207989,3785.390742,-2.438943e+04,111430.271429,0.000000,107438.33,109255.99,108271.49,108958.04,688.20
2025-07-09,101285.329040,115642.848960,52.0,5.365831,7683.832396,-2.060404e+04,111430.271429,0.000000,108329.87,112152.91,108953.58,111282.85,2324.81


# XGBoost Implementation

In [ ]:
merged['close'].plot.line()
plt.title(f'Price Plot for {COIN}')
plt.xlabel('Date')
plt.ylabel('Price')
# format y axis to show currency
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: "${:,}".format(int(x))))

In [ ]:
scaler = StandardScaler()
closeScaler = StandardScaler()
mergedIndex = merged.index
mergedCols = merged.columns

mergedWithoutClose = merged.drop(columns=['close'])
mwocCols = mergedWithoutClose.columns
mwocIndex = mergedWithoutClose.index
mergedWithoutClose = scaler.fit_transform(mergedWithoutClose)
mergedWithoutClose = pd.DataFrame(mergedWithoutClose, index=mergedIndex, columns=mwocCols)

merged['close'] = closeScaler.fit_transform(merged['close'].values.reshape(-1, 1))
merged = pd.concat([mergedWithoutClose, merged['close']], axis=1)
merged

In [ ]:
training_cols = trainingCols()
X = merged[training_cols]
y = merged['close']
n = len(data)

merged = transformerDataSetup(merged)
X_train, X_test, y_train, y_test, _, _ = transformerXTrainYTrain(merged, testSize=len(merged)-TEST_DAYS)
train_stuff = merged.loc[X_train.index, training_cols]
test_stuff = merged.loc[X_test.index, training_cols]
X_train_norm = pd.concat([X_train, train_stuff], axis=1)
X_test_norm = pd.concat([X_test, test_stuff], axis=1)

X_train_norm.shape, X_test_norm.shape, y_train.shape, y_test.shape

In [ ]:
xgb_model = XGBoost(
    base_estimator=RandomForestRegressor(n_estimators=50),
    n_estimators=50,
    learning_rate=0.1,
    max_depth=8,
    random_state=42,
)

In [ ]:
xgb_model.fit(X_train_norm, y_train)
with open(f'models/{COIN}_full_xgb_model_everything.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

In [ ]:
predictions = xgb_model.predict(X_test_norm.iloc[[-1]]) # Predicting for the future 7 days
modelName = 'full_xgb_model_everything'
predictions = np.array(predictions)
predictions = predictions.reshape(-1, 1)
predictions = closeScaler.inverse_transform(predictions)
predictions = pd.DataFrame(predictions, columns=['predictions'])
predictions.index = pd.date_range(start=merged.index.max(), periods=TEST_DAYS, freq='D')
if not os.path.exists(f'predictions/{COIN}'):
    os.makedirs(f'predictions/{COIN}')
predictions.to_csv(f'predictions/{COIN}/{modelName}_predictions.csv')
predictions.plot.line()
plt.title(f'XGBoost Predictions for {COIN}')
plt.xlabel('Date')
plt.ylabel('Price')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: "${:,}".format(int(x))))
plt.show()

In [ ]:
predictions_1 = xgb_model.predict(X_test_norm.iloc[[-2]])
predictions_1 = np.array(predictions_1)
predictions_1 = predictions_1.reshape(-1, 1)
predictions_1 = closeScaler.inverse_transform(predictions_1)
predictions_1 = pd.DataFrame(predictions_1, columns=['predictions'])
predictions_1['actual'] = closeScaler.inverse_transform(np.array(y_test.iloc[-2]).reshape(-1, 1))

predictions_1.index = pd.date_range(merged.index.max()-pd.Timedelta(days=1), periods=len(predictions_1), freq='D')
predictions_1.plot.line()
mse = mean_squared_error(predictions_1['actual'], predictions_1['predictions'], squared=False)
plt.title(f'Predictions vs Actual for {COIN} - Last 2 Days\nRMSE = ${mse:.2f}')
plt.show()